# 🛠️ Local AI Coding Loop on Colab (A100 / H100)

Run a **fully local, agentic coding loop** — no API keys, no external model calls.
A HuggingFace open-weight coding model is served with **vLLM** and driven by the
**OpenHands** autonomous coding agent.

```
HuggingFace weights ──▶ vLLM server ──▶ OpenAI API ──▶ OpenHands loop ──▶ /workspace
     (model)            (port 8000)     (tool calls)   (CodeActAgent)     (your code)
```

Optimized for quality + speed on one 80 GB GPU: **FP8 on H100**, **BF16 on A100**,
prefix caching, and a fast MoE default model (~3B active params).

**Steps:** set runtime to an A100/H100 GPU, then run every cell top-to-bottom.
Edit the **task** cell and re-run it to iterate.

> ⚠️ **Security:** OpenHands' Local Runtime has **no sandbox** — the agent can run
> arbitrary commands and edit any file on this VM. Fine on a disposable Colab VM;
> never point it at a machine with secrets you care about.

## 1. Setup — fetch helpers & enable nested event loops

In [ ]:
#@title Setup { display-mode: "form" }
#@markdown Makes `helpers.py` importable. If the notebook was opened standalone
#@markdown (e.g. via GitHub), it clones the repo folder. For a **private** repo,
#@markdown add a token to the URL or upload `helpers.py` next to this notebook.
REPO_URL = "https://github.com/engineer21-a/claude-code.git"  #@param {type:"string"}
BRANCH   = "claude/colab-local-ai-coding-loop-u1uq1d"  #@param {type:"string"}

import os, sys, subprocess

if not os.path.exists("helpers.py"):
    if not os.path.exists("claude-code"):
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL], check=True)
    os.chdir("claude-code/colab-local-coding-agent")

sys.path.insert(0, os.getcwd())
print("Working dir:", os.getcwd())
assert os.path.exists("helpers.py"), "helpers.py not found — clone the repo or upload helpers.py here."

## 2. Detect GPU → choose FP8 vs BF16

In [ ]:
import helpers

gpu = helpers.detect_gpu()
print(gpu.summary)
if gpu.memory_gb and gpu.memory_gb < 70:
    print("\n⚠️  This GPU has < 80 GB. Use an A100/H100 80 GB runtime, or lower")
    print("    MAX_MODEL_LEN / GPU_MEM_UTIL below and expect possible OOM.")

## 3. Configure the model & run parameters

In [ ]:
#@title Configuration { display-mode: "form" }
#@markdown Default = OpenHands' recommended local model. See README for the full table.
MODEL = "Qwen/Qwen3.6-35B-A3B"  #@param ["Qwen/Qwen3.6-35B-A3B", "Qwen/Qwen3-Coder-30B-A3B-Instruct", "Qwen/Qwen3-Coder-Next", "mistralai/Devstral-Small-2507"] {allow-input: true}
MAX_MODEL_LEN = 65536  #@param {type:"integer"}
GPU_MEM_UTIL = 0.92  #@param {type:"number"}
TENSOR_PARALLEL = 1  #@param {type:"integer"}
MAX_ITERATIONS = 50  #@param {type:"integer"}
HF_TOKEN = ""  #@param {type:"string"}
API_KEY = "local-key"

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
print(f"Model: {MODEL}")
print(f"Tool-call parser: {helpers.tool_parser_for(MODEL)}  |  precision: {gpu.recommended_quantization or 'BF16'}")

## 4. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

# Colab runs inside an event loop; vLLM/OpenHands need nested loops to work.
import nest_asyncio
nest_asyncio.apply()
print("Dependencies installed and nest_asyncio applied ✅")

## 5. Launch the vLLM server (background) and wait until ready
First start downloads weights + loads the model — this can take several minutes.
Logs stream to `/content/vllm_server.log`.

In [ ]:
helpers.start_vllm(
    MODEL,
    gpu=gpu,
    api_key=API_KEY,
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_MEM_UTIL,
    tensor_parallel_size=TENSOR_PARALLEL,
)
helpers.wait_for_server(api_key=API_KEY, timeout=1800)

## 6. Point OpenHands at the local server
Uses the **Local Runtime** (`RUNTIME=local`) so no Docker is required.

In [ ]:
WORKSPACE = "/content/workspace"  #@param {type:"string"}

env = helpers.configure_openhands(MODEL, api_key=API_KEY, workspace=WORKSPACE)
for k, v in env.items():
    print(f"{k}={v}")

## 7. (Optional) bring in a project to work on
Clone an existing repo into the workspace, or skip this to start from scratch.
Initialize git so you can review the agent's diffs.

In [ ]:
PROJECT_GIT_URL = ""  #@param {type:"string"}

if PROJECT_GIT_URL:
    !rm -rf "$WORKSPACE" && git clone "$PROJECT_GIT_URL" "$WORKSPACE"
else:
    os.makedirs(WORKSPACE, exist_ok=True)
    if not os.path.isdir(os.path.join(WORKSPACE, ".git")):
        !cd "$WORKSPACE" && git init -q && git commit -q --allow-empty -m "baseline"
print("Workspace ready:", WORKSPACE)

## 8. ▶️ Run a coding task — the agentic loop
Describe what you want. The agent reads/edits/runs code until it's done. **Re-run
this cell with a new task to keep iterating.**

In [ ]:
#@title Coding task { display-mode: "form" }
TASK = "Create fizzbuzz.py with a fizzbuzz(n) function and a pytest test file, then run the tests and make them pass."  #@param {type:"string"}

exit_code = helpers.run_openhands_task(TASK, max_iterations=MAX_ITERATIONS)
print(f"\nAgent finished with exit code {exit_code}")

## 9. Review what the agent changed

In [ ]:
!cd "$WORKSPACE" && git add -A && git status && echo '----- DIFF -----' && git diff --cached

## 10. Save your work & free the GPU
Colab VMs are ephemeral — **push or download `/content/workspace` before you
disconnect.** `stop_vllm()` frees the GPU when you're done.

In [ ]:
# Example: zip the workspace to download via the Files panel.
# !cd /content && zip -qr workspace.zip workspace && echo 'wrote /content/workspace.zip'

helpers.stop_vllm()